# Regime-Aware Pairs Price Generator
### OU spread simulation with HMM regime labels for PPO training

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
from hmmlearn import hmm
import yfinance as yf
from pathlib import Path
from statsmodels.tools import add_constant
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.stattools import adfuller, coint

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})


TICKER_A     = os.environ.get('GEN_A',     'V')
TICKER_B     = os.environ.get('GEN_B',     'MA')
START_DATE   = os.environ.get('GEN_START', '2009-01-01')
END_DATE     = os.environ.get('GEN_END',   '2022-12-31')

N_SCENARIOS  = 3_000
HORIZON      = 252
DT           = 1 / 252
N_REGIMES    = 3
HMM_WINDOW   = 20
HMM_RESTARTS = 20

RANDOM_SEED  = 42
PAIR_NAME = f"{TICKER_A}_{TICKER_B}"
OUT_DIR   = Path(f"regime_output_{PAIR_NAME}")

REGIME_NAMES   = {0: 'stable', 1: 'neutral', 2: 'crisis'}
REGIME_COLORS  = {0: '#2196F3', 1: '#FF9800', 2: '#E91E63'}


np.random.seed(RANDOM_SEED)
OUT_DIR.mkdir(exist_ok=True)
print(f'Calibrating {PAIR_NAME}  {START_DATE} -> {END_DATE}')

## 1. Download Data

In [ ]:
raw = yf.download([TICKER_A, TICKER_B], start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)['Close'].dropna()
print(f'{len(raw)} rows  ({raw.index[0].date()} -> {raw.index[-1].date()})')
print(f'{TICKER_A}: ${raw[TICKER_A].min():.1f} – ${raw[TICKER_A].max():.1f}')
print(f'{TICKER_B}: ${raw[TICKER_B].min():.1f} – ${raw[TICKER_B].max():.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6), sharex=True)
ax.plot(raw.index, raw[TICKER_A], color='blue', lw=1, label=TICKER_A)
ax.plot(raw.index, raw[TICKER_B], color='red', lw=1, label=TICKER_B)

ax.set_title(f'Historical Prices: {TICKER_A} & {TICKER_B}', fontsize=13)
ax.set_ylabel('Price ($)')
ax.set_xlabel('Date')

ax.legend(loc='upper left')
plt.tight_layout()
plt.show()


## 2. Find the Best Cointegrated Sub-Window

Scan all start/end year combinations (min 3 years). Pick the window with the lowest ADF p-value on the OLS log-spread. All OU and HMM calibration happens on this window only.

In [ ]:
log_A_full = np.log(raw[TICKER_A])
log_B_full = np.log(raw[TICKER_B])
dates = raw.index
years = sorted(set(dates.year))
results = []

for i, y0 in enumerate(years):
    for y1 in years[i + 3:]:
        mask = (dates.year >= y0) & (dates.year <= y1)
        lA = log_A_full[mask].values
        lB = log_B_full[mask].values
        if len(lA) < 252:
            continue
        ols = OLS(lA, add_constant(lB)).fit()
        spread = lA - ols.params[1] * lB
        try:
            adf_p = adfuller(spread, maxlag=1, autolag=None)[1]
        except Exception:
            continue
        results.append((adf_p, y0, y1, ols.params[1], len(lA)))

results.sort()
print(f"{'Start':>6}  {'End':>5}  {'N':>5}  {'Beta':>7}  {'ADF-p':>7}  Status")
print('-' * 50)
for adf_p, y0, y1, beta_val, n in results[:10]:
    print(f'{y0:>6}  {y1:>5}  {n:>5}  {beta_val:>7.4f}  {adf_p:>7.4f}')
    flag = '✓' if adf_p < 0.05 else ('~' if adf_p < 0.10 else '✗')
    print(f'{y0:>6}  {y1:>5}  {n:>5}  {beta_val:>7.4f}  {adf_p:>7.4f}  {flag}')

best_adf_p, best_y0, best_y1, best_beta, _ = results[0]
print(f'\n→ Using: {best_y0}–{best_y1}  beta={best_beta:.4f}  ADF p={best_adf_p:.4f}')

mask_best  = (dates.year >= best_y0) & (dates.year <= best_y1)
log_A      = log_A_full[mask_best].values
log_B      = log_B_full[mask_best].values
log_spread = log_A - best_beta * log_B
dates_best = dates[mask_best]

In [ ]:
# Plot: full-history spread with calibration window shaded
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('Log Prices & Spread — Calibration Window Highlighted', fontsize=12)

axes[0].plot(raw.index, log_A_full, color='steelblue', lw=0.9, label=f'log {TICKER_A}')
axes[0].plot(raw.index, log_B_full, color='tomato',    lw=0.9, label=f'log {TICKER_B}')
axes[0].axvspan(dates_best[0], dates_best[-1], alpha=0.12, color='green',
                label=f'Calibration {best_y0}–{best_y1}')
axes[0].set_ylabel('Log Price'); axes[0].legend(fontsize=9)

spread_full = log_A_full.values - best_beta * log_B_full.values
axes[1].plot(raw.index, spread_full, color='purple', lw=0.8, alpha=0.8)
axes[1].axvspan(dates_best[0], dates_best[-1], alpha=0.12, color='green')
axes[1].axhline(log_spread.mean(), color='black', lw=1, ls='--',
                label=f'Calib. mean={log_spread.mean():.4f}')
axes[1].set_ylabel(f'log {TICKER_A} − β·log {TICKER_B}')
axes[1].set_xlabel('Date'); axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Fit the OU Process

In [ ]:
sx = log_spread[:-1]
sy = log_spread[1:]

def neg_ll(params):
    kappa, mu, sigma = params
    if kappa <= 0 or sigma <= 0:
        return 1e12
    e   = np.exp(-kappa * DT)
    c_m = sx * e + mu * (1 - e)
    c_v = (sigma**2 / (2 * kappa)) * (1 - e**2)
    if c_v <= 0:
        return 1e12
    return -norm.logpdf(sy, c_m, np.sqrt(c_v)).sum()

coef  = np.polyfit(sx, sy - sx, 1)[0]
k0    = max(-coef / DT, 0.5) if coef < 0 else 1.5
m0    = float(log_spread.mean())
s0    = float(log_spread.std()) * np.sqrt(2 * k0)

res   = minimize(neg_ll, [k0, m0, s0],
                     bounds=[(0.01, 50), (None, None), (1e-6, None)],
                     method='L-BFGS-B')
kappa, mu_ou, sigma_global = res.x
half_life = np.log(2) / kappa * 252
phi_disc  = float(np.exp(-kappa * DT))
e_g       = phi_disc
sigma_stat = sigma_global / np.sqrt(2 * kappa)

print(f'κ (mean-reversion speed) = {kappa:.4f}')
print(f'μ (long-run mean)         = {mu_ou:.5f}')
print(f'σ_global (volatility)     = {sigma_global:.5f}')
print(f'Half-life                 = {half_life:.1f} trading days')
print(f'φ (discrete AR coeff)     = {phi_disc:.6f}')
print(f'Horizon                   = {HORIZON} steps  (~{HORIZON/252:.1f} yr,  ~{HORIZON/half_life:.1f} HL cycles)')

## 4. Fit HMM - Regime Detection

In [ ]:
s_ser  = pd.Series(log_spread, index=dates_best)
mu_r   = s_ser.rolling(HMM_WINDOW).mean()
std_r  = s_ser.rolling(HMM_WINDOW).std().replace(0, np.nan)

# f1: rolling z-score
zscore_feat = ((s_ser - mu_r) / std_r)

# f2: log-volatility
vol_raw     = s_ser.diff().rolling(HMM_WINDOW).std() * np.sqrt(252)
logvol_feat = np.log(vol_raw.clip(lower=1e-6))

# f3: realised mean-reversion rate over 60-day rolling window
REV_WINDOW    = 60
REV_THRESHOLD = 1.0

def reversion_rate_series(series, window=REV_WINDOW, threshold=REV_THRESHOLD):
    vals   = series.values
    n      = len(vals)
    out    = np.full(n, np.nan)
    for i in range(window, n):
        w   = vals[max(0, i - window): i + 1]
        mu  = w.mean()
        std = w.std() + 1e-8
        n_sig, n_rev = 0, 0
        for j in range(len(w) - 1):
            if abs((w[j] - mu) / std) > threshold:
                n_sig += 1
                if (w[j+1] - w[j]) * (mu - w[j]) > 0:
                    n_rev += 1
        out[i] = n_rev / n_sig if n_sig > 0 else 0.5
    return pd.Series(out, index=series.index)

rev_rate_raw = reversion_rate_series(s_ser)
rev_clipped  = rev_rate_raw.clip(lower=0.05, upper=0.95)
logit_rev_feat = np.log(rev_clipped / (1 - rev_clipped))


common_idx = (zscore_feat.dropna().index
              .intersection(logvol_feat.dropna().index)
              .intersection(logit_rev_feat.dropna().index))

feat = np.column_stack([
    zscore_feat[common_idx].values,
    logvol_feat[common_idx].values,
    logit_rev_feat[common_idx].values,
])

print(f'feat shape      : {feat.shape}')
print(f'feat NaN count  : {np.isnan(feat).sum()}')
print(f'zscore  range   : [{feat[:,0].min():.3f}, {feat[:,0].max():.3f}]')
print(f'logvol  range   : [{feat[:,1].min():.3f}, {feat[:,1].max():.3f}]')
print(f'logit_r range   : [{feat[:,2].min():.3f}, {feat[:,2].max():.3f}]')

feat_sc     = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
dates_feat  = common_idx
spread_feat = log_spread[s_ser.index.get_indexer(dates_feat)]

# Fit HMM with multiple restarts
best_hmm, best_score = None, -np.inf
for seed in range(HMM_RESTARTS):
    m = hmm.GaussianHMM(n_components=N_REGIMES, covariance_type='full',
                         n_iter=500, tol=1e-6, random_state=seed)
    try:
        m.fit(feat_sc)
        sc = m.score(feat_sc)
        if sc > best_score:
            best_score, best_hmm = sc, m
    except Exception:
        pass

raw_labels = best_hmm.predict(feat_sc)
trans_raw  = best_hmm.transmat_

med_vol = {r: np.median(feat[raw_labels == r, 1]) for r in range(N_REGIMES)}
order   = sorted(med_vol, key=med_vol.get)
remap   = {order[0]: 0, order[1]: 1, order[2]: 2}
labels  = np.array([remap[l] for l in raw_labels])


trans = np.zeros((3, 3))
for old_i, new_i in remap.items():
    for old_j, new_j in remap.items():
        trans[new_i, new_j] = trans_raw[old_i, old_j]

counts = {REGIME_NAMES[r]: int((labels == r).sum()) for r in range(3)}
print('Regime counts:', counts)
print()
print('Transition matrix (rows=from, cols=to):')
print(f"{'':12s}  {'stable':>8}  {'neutral':>8}  {'crisis':>8}")
for i in range(3):
    row = '  '.join(f'{trans[i,j]:>8.4f}' for j in range(3))
    print(f"  {REGIME_NAMES[i]:10s}  {row}")
print()

for r in range(3):
    mask = labels == r
    rr   = np.exp(feat[mask,2]) / (1 + np.exp(feat[mask,2]))  # back to rate
    print(f"  {REGIME_NAMES[r]:8s}: logvol={feat[mask,1].mean():.3f}  "
          f"rev_rate={rr.mean():.3f}")


from copy import deepcopy
hmm_canon            = deepcopy(best_hmm)
hmm_canon.means_     = best_hmm.means_[order]
hmm_canon.covars_    = best_hmm.covars_[order]
hmm_canon.startprob_ = best_hmm.startprob_[order]
hmm_canon.transmat_  = best_hmm.transmat_[np.ix_(order, order)]
best_hmm             = hmm_canon
print('HMM rearranged — means_[0]=stable, [1]=neutral, [2]=crisis')
for r in range(3):
    print(f'  {REGIME_NAMES[r]:8s}: {best_hmm.means_[r]}')


from numpy.linalg import eig as _eig
_ev, _evec = _eig(trans.T)
_idx = np.argmin(np.abs(_ev - 1.0))
_st  = np.real(_evec[:, _idx])
stationary_dist = (_st / _st.sum()).astype(np.float32)
print(f'Stationary dist: {stationary_dist}')

In [ ]:
# Plot: spread coloured by regime
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Calibration Spread — HMM Regime Labels', fontsize=12)

# Top: spread with regime background shading
axes[0].plot(dates_feat, spread_feat, color='#444', lw=0.8, zorder=2)
axes[0].axhline(mu_ou, color='black', lw=1, ls='--', alpha=0.5, label='μ')

# shade regime periods
prev_r, t0 = labels[0], dates_feat[0]
for i in range(1, len(labels)):
    if labels[i] != prev_r or i == len(labels)-1:
        axes[0].axvspan(t0, dates_feat[i],
                        color=REGIME_COLORS[prev_r], alpha=0.20)
        t0, prev_r = dates_feat[i], labels[i]

from matplotlib.patches import Patch
patches = [Patch(color=REGIME_COLORS[r], alpha=0.5, label=REGIME_NAMES[r].capitalize())
           for r in range(3)]
axes[0].legend(handles=patches, fontsize=9)
axes[0].set_ylabel('Log Spread')

# Bottom: regime label timeline
axes[1].scatter(dates_feat, labels,
                c=[REGIME_COLORS[l] for l in labels], s=3, zorder=2)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['Stable', 'Neutral', 'Crisis'])
axes[1].set_xlabel('Date')

plt.tight_layout(); plt.show()

In [ ]:
# Plot: HMM transition matrix heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(trans, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
tick_labels = ['Stable', 'Neutral', 'Crisis']
ax.set_xticks([0,1,2]); ax.set_xticklabels(tick_labels)
ax.set_yticks([0,1,2]); ax.set_yticklabels(tick_labels)
ax.set_xlabel('To'); ax.set_ylabel('From')
ax.set_title('Transition Probability Matrix', fontsize=11)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{trans[i,j]:.3f}', ha='center', va='center',
                color='white' if trans[i,j] > 0.6 else 'black', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Per-Regime σ Estimation

In [ ]:
mu_r, kappa_r, sigma_r = {}, {}, {}

print(f'Global fallback: kappa={kappa:.3f}  mu={mu_ou:.5f}  sigma={sigma_global:.5f}  (used if n<30)')
print()

for r in range(N_REGIMES):
    s_arr  = spread_feat
    mask_r = (labels[:-1] == r) & (labels[1:] == r)
    sx_r   = s_arr[:-1][mask_r]
    sy_r   = s_arr[1:][mask_r]
    n_r    = len(sx_r)
    seg    = spread_feat[labels == r]

    if n_r < 30:
        mu_r[r], kappa_r[r], sigma_r[r] = mu_ou, kappa, sigma_global
        note = f'n={n_r} too small -> using global params'
    else:
        
        mu_r[r] = mu_ou +  (float(seg.mean()) - mu_ou)
        c_coef     = np.polyfit(sx_r, sy_r - sx_r, 1)[0]
        kappa_r[r] = float(np.clip(-c_coef / DT, 0.1, 50.0)) if c_coef < 0 else kappa
        e_r        = np.exp(-kappa_r[r] * DT)
        var_fac_r  = (1.0 - e_r**2) / (2.0 * kappa_r[r])
        c_m        = sx_r * e_r + mu_r[r] * (1 - e_r)
        sigma_r[r] = float(np.sqrt(np.mean((sy_r - c_m)**2) / var_fac_r))
        note = f'n={n_r}'

    hl_r = np.log(2) / kappa_r[r] * 252
    print(f'  Regime {r} ({REGIME_NAMES[r]:7s}): mu={mu_r[r]:.4f} (dev={mu_r[r]-mu_ou:+.4f})  '
          f'kappa={kappa_r[r]:.2f} (HL={hl_r:.0f}d)  sigma={sigma_r[r]:.5f}  {note}')

# Sanity check: crisis sigma should be the largest
ordered_ok = sigma_r[2] >= sigma_r[1] >= sigma_r[0]
print()
print('sigma ordering:', 'OK' if ordered_ok else 'check HMM')

## 6. Simulate Spread Paths with Regime Switching

In [ ]:
np.random.seed(RANDOM_SEED)  # ensure reproducibility
cum_trans = np.cumsum(trans, axis=1)  # for fast vectorised regime transition

# Per-regime discrete OU coefficients (each regime has its own kappa, mu, sigma)
e_g_r   = {r: float(np.exp(-kappa_r[r] * DT)) for r in range(N_REGIMES)}
c_std_r = {
    r: float(np.sqrt((sigma_r[r]**2 / (2 * kappa_r[r])) * (1 - e_g_r[r]**2)))
    for r in range(N_REGIMES)
}

spreads = np.zeros((N_SCENARIOS, HORIZON), dtype=np.float32)
regimes = np.zeros((N_SCENARIOS, HORIZON), dtype=np.int8)

init_spread_0 = float(
    np.log(raw[TICKER_A].iloc[-1]) - best_beta * np.log(raw[TICKER_B].iloc[-1])
)
spreads[:, 0] = init_spread_0
regimes[:, 0] = int(labels[-1])   # start from last observed regime

for t in range(1, HORIZON):
    prev_r = regimes[:, t-1].astype(int)
    rnd    = np.random.rand(N_SCENARIOS)

    # Vectorised Markov transition
    new_r = np.zeros(N_SCENARIOS, dtype=int)
    for r in range(N_REGIMES):
        mask = prev_r == r
        if mask.any():
            new_r[mask] = (rnd[mask, None] > cum_trans[r, :-1]).sum(axis=1)
    regimes[:, t] = new_r

    eg     = np.array([e_g_r[r] for r in new_r])
    mu_now = np.array([mu_r[r]  for r in new_r])
    c_mean = spreads[:, t-1] * eg + mu_now * (1 - eg)
    noise  = np.zeros(N_SCENARIOS)
    for r in range(N_REGIMES):
        mask = new_r == r
        if mask.any():
            noise[mask] = np.random.normal(0, c_std_r[r], mask.sum())
    spreads[:, t] = c_mean + noise

# Regime frequency check
print('Regime frequencies in generated data:')
hist_freq = {REGIME_NAMES[r]: (labels == r).mean() for r in range(N_REGIMES)}
gen_freq  = {REGIME_NAMES[r]: (regimes == r).mean() for r in range(N_REGIMES)}
print(f"{'Regime':10s}  {'Historical':>12}  {'Generated':>12}")
for r in range(N_REGIMES):
    print(f"  {REGIME_NAMES[r]:10s}  {hist_freq[REGIME_NAMES[r]]:>12.1%}  "
          f"{gen_freq[REGIME_NAMES[r]]:>12.1%}")

In [ ]:
# Plot: 6 individual paths coloured by regime
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Sample Spread Paths — Coloured by Regime', fontsize=12)

for idx, ax in enumerate(axes.flat):
    path = spreads[idx]
    regs = regimes[idx]
    ax.axhline(mu_ou, color='gray', lw=0.8, ls='--')
    # colour segments by regime
    for t in range(HORIZON - 1):
        ax.plot([t, t+1], [path[t], path[t+1]],
                color=REGIME_COLORS[regs[t]], lw=1.0)
    ax.set_title(f'Path {idx+1}')
    ax.set_xlabel('Day'); ax.set_ylabel('Log Spread')

from matplotlib.lines import Line2D
legend_lines = [Line2D([0],[0], color=REGIME_COLORS[r], lw=2,
                        label=REGIME_NAMES[r].capitalize()) for r in range(3)]
fig.legend(handles=legend_lines, loc='lower center', ncol=3, fontsize=10)
plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()

## 7. Reconstruct Individual Prices

In [ ]:
np.random.seed(RANDOM_SEED)  # ensure reproducibility
b  = abs(best_beta)
b2 = 1.0 + b**2

# Common trend drift from calibration window
c_hist = (b * log_A + log_B) / (1.0 + b)
dc     = np.diff(c_hist)
mu_c   = dc.mean()
sig_c  = dc.std()
print(f'b = {b:.4f}')
print(f'Common trend: μ={mu_c:.6f}/day  σ={sig_c:.6f}/day')

init_log_A = float(np.log(raw[TICKER_A].iloc[-1]))
init_log_B = float(np.log(raw[TICKER_B].iloc[-1]))

log_A_gen = np.zeros((N_SCENARIOS, HORIZON))
log_B_gen = np.zeros((N_SCENARIOS, HORIZON))
log_A_gen[:, 0] = init_log_A
log_B_gen[:, 0] = init_log_B

for t in range(1, HORIZON):
    trend = np.random.normal(mu_c, sig_c, N_SCENARIOS)
    c_prev = (b * log_A_gen[:, t-1] + log_B_gen[:, t-1]) / (1.0 + b)
    c      = c_prev + trend
    u      = c * (1.0 + b)
    z      = spreads[:, t]
    log_A_gen[:, t] = (z + b * u) / b2
    log_B_gen[:, t] = (u - b * z) / b2

prices_A = np.exp(log_A_gen)
prices_B = np.exp(log_B_gen)

# Reconstruction check
z_check = log_A_gen[:, 1] - b * log_B_gen[:, 1]
err     = np.max(np.abs(z_check - spreads[:, 1]))
print(f'Reconstruction error: {err:.2e}  {"✓" if err < 1e-10 else "✗"}')
print(f'{TICKER_A}  mean=${prices_A.mean():.2f}  std=${prices_A.std():.2f}')
print(f'{TICKER_B}  mean=${prices_B.mean():.2f}  std=${prices_B.std():.2f}')

## 8. Validate

We generate a **separate validation batch** at the theoretically correct
horizon and run all statistical checks on that. The training data saved still uses HORIZON=252.

In [ ]:
np.random.seed(RANDOM_SEED)  # ensure reproducibility
# -- Compute the minimum horizon for ADF to have power (uses global kappa) --
phi_disc      = float(np.exp(-kappa * DT))
T_min_adf     = int((2.87**2) * (1 - phi_disc**2) / (1 - phi_disc)**2 * 2)
HORIZON_VAL   = max(T_min_adf, int(10 * half_life), 756)
print(f'Training horizon  : {HORIZON} steps  ({HORIZON/252:.1f} yr)')
print(f'Validation horizon: {HORIZON_VAL} steps  ({HORIZON_VAL/252:.1f} yr)')
print(f'Reason: need T >= {T_min_adf} for ADF power at phi={phi_disc:.4f}')

# -- Generate a validation batch at the longer horizon (per-regime kappa/mu/sigma) --
N_VAL         = 300
spreads_val   = np.zeros((N_VAL, HORIZON_VAL))
regimes_val   = np.zeros((N_VAL, HORIZON_VAL), dtype=np.int8)
spreads_val[:, 0] = init_spread_0
regimes_val[:, 0] = int(labels[-1])

for t in range(1, HORIZON_VAL):
    prev_r = regimes_val[:, t-1].astype(int)
    rnd    = np.random.rand(N_VAL)
    new_r  = np.zeros(N_VAL, dtype=int)
    for r in range(N_REGIMES):
        mask = prev_r == r
        if mask.any():
            new_r[mask] = (rnd[mask, None] > cum_trans[r, :-1]).sum(axis=1)
    regimes_val[:, t] = new_r
    eg     = np.array([e_g_r[r] for r in new_r])
    mu_now = np.array([mu_r[r]  for r in new_r])
    c_mean = spreads_val[:, t-1] * eg + mu_now * (1 - eg)
    noise  = np.zeros(N_VAL)
    for r in range(N_REGIMES):
        mask = new_r == r
        if mask.any():
            noise[mask] = np.random.normal(0, c_std_r[r], mask.sum())
    spreads_val[:, t] = c_mean + noise

# Reconstruct log prices for validation batch
log_A_val = np.zeros((N_VAL, HORIZON_VAL))
log_B_val = np.zeros((N_VAL, HORIZON_VAL))
log_A_val[:, 0] = init_log_A
log_B_val[:, 0] = init_log_B

for t in range(1, HORIZON_VAL):
    trend  = np.random.normal(mu_c, sig_c, N_VAL)
    c_prev = (b * log_A_val[:, t-1] + log_B_val[:, t-1]) / (1.0 + b)
    c      = c_prev + trend
    u      = c * (1.0 + b)
    z      = spreads_val[:, t]
    log_A_val[:, t] = (z + b * u) / b2
    log_B_val[:, t] = (u - b * z) / b2

# -- Run statistical checks on validation batch --
n_coint   = 0
n_adf     = 0
halflives = []

for i in range(N_VAL):
    lA_i = log_A_val[i]
    lB_i = log_B_val[i]
    ls_i = lA_i - b * lB_i
    try:
        _, pv, _ = coint(lA_i, lB_i, maxlag=1)
        if pv < 0.05: n_coint += 1
    except Exception: pass
    try:
        if adfuller(ls_i, maxlag=1, autolag=None)[1] < 0.05: n_adf += 1
    except Exception: pass
    try:
        c_coef = np.polyfit(ls_i[:-1], np.diff(ls_i), 1)[0]
        if c_coef < 0:
            hl = -np.log(2) / c_coef
            if 1 < hl < 5000: halflives.append(hl)
    except Exception: pass

avg_hl = np.mean(halflives) if halflives else float('nan')
hl_err = abs(avg_hl - half_life) / half_life * 100 if not np.isnan(avg_hl) else float('nan')

sep  = '=' * 56
dash = '-' * 56
ok   = lambda v: 'OK' if v else 'X'
print(sep)
print('VALIDATION REPORT  (long-horizon batch for test power)')
print(sep)
print(f'Validation scenarios : {N_VAL}')
print(f'Validation horizon   : {HORIZON_VAL} steps  ({HORIZON_VAL/252:.1f} yr)')
print(f'Training horizon     : {HORIZON} steps  ({HORIZON/252:.1f} yr)  [saved separately]')
print(f'Fitted half-life     : {half_life:.1f} days')
print(dash)
print(f'% cointegrated       : {n_coint/N_VAL*100:.1f}%  ' + ok(n_coint/N_VAL > 0.8))
print(f'% ADF stationary     : {n_adf/N_VAL*100:.1f}%  '   + ok(n_adf/N_VAL   > 0.8))
print(f'Avg generated HL     : {avg_hl:.1f} days')
print(f'HL error vs fitted   : {hl_err:.1f}%  '             + ok(hl_err < 40))
print(sep)


## 9. Sanity-Check Plots

In [ ]:
# ADF p-value distribution — on validation batch (long horizon)
adf_pvals = []
for i in range(N_VAL):
    ls_i = log_A_val[i] - b * log_B_val[i]
    try:
        adf_pvals.append(adfuller(ls_i, maxlag=1, autolag=None)[1])
    except Exception:
        pass

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(adf_pvals, bins=40, color='green', alpha=0.6, edgecolor='none')
ax.axvline(0.05, color='red', lw=2, ls='--', label='5% threshold')
pct_pass = np.mean(np.array(adf_pvals) < 0.05) * 100
ax.text(0.98, 0.95, f'{pct_pass:.1f}% pass at 5%',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        color='green' if pct_pass > 80 else 'red')
ax.set_title(
    f'ADF p-value Distribution — Validation Batch\n'
    f'(T={HORIZON_VAL} steps; training episodes use T={HORIZON})',
    fontsize=11)
ax.set_xlabel('ADF p-value'); ax.set_ylabel('Count')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 10. Save

In [ ]:
np.savez_compressed(
    OUT_DIR / 'generated.npz',
    prices_A = prices_A,    # (N_SCENARIOS, HORIZON) raw prices
    prices_B = prices_B,
    log_A    = log_A_gen,   # log prices
    log_B    = log_B_gen,
    spreads  = spreads,     # log spread  — main PPO observation
    regimes  = regimes,     # int8: 0=stable 1=neutral 2=crisis
)
print(f'Saved → {OUT_DIR / "generated.npz"}')
print(f'Shapes:')
print(f'  prices_A : {prices_A.shape}')
print(f'  prices_B : {prices_B.shape}')
print(f'  spreads  : {spreads.shape}')
print(f'  regimes  : {regimes.shape}  (0=stable, 1=neutral, 2=crisis)')
print()
print('Load with:')
print("  d = np.load('regime_output/generated.npz')")
print("  pA      = d['prices_A']   # (1000, 252)")
print("  pB      = d['prices_B']")
print("  spreads = d['spreads']")
print("  regimes = d['regimes']    # feed to PPO as observation")

In [ ]:
import joblib

joblib.dump(best_hmm, OUT_DIR / 'hmm_model.pkl')

np.save(
    OUT_DIR / 'feat_scaler.npy',
    np.array([feat.mean(0), feat.std(0)])  # shape (2, 3): means and stds for 3 features
)

np.save(OUT_DIR / 'beta.npy', np.array([best_beta]))

print(f'Saved:')
print(f'  hmm_model.pkl      -> HMM for online regime inference')
print(f'  feat_scaler.npy    -> feature means and stds')
print(f'  beta.npy           -> beta_for_generation = {best_beta:.4f}')

import pickle
ou_params = {
    'kappa'        : kappa,
    'mu_ou'        : mu_ou,
    'sigma_global' : sigma_global,
    'sigma_r'      : sigma_r,
    'mu_r'         : mu_r,
    'kappa_r'      : kappa_r,
    'trans'        : trans,
    'init_spread'  : init_spread_0,
    'DT'           : DT,
    'N_REGIMES'    : N_REGIMES,
    'HORIZON'      : HORIZON,
    'REGIME_NAMES' : REGIME_NAMES,
}
with open(OUT_DIR / 'ou_params.pkl', 'wb') as _f:
    pickle.dump(ou_params, _f)
print('Saved ou_params.pkl  (with per-regime mu_r, kappa_r, sigma_r)')
